# Lab 3 · Transformation: bronze to silver to gold

Clean a messy bronze table into silver, join and union it, count nulls correctly, upsert with MERGE, see why INSERT INTO duplicates on rerun, keep rows safely with DELETE, and publish gold as a view, a table and a materialized view.

**Exam:** Databricks Certified Data Engineer Associate · **Domain:** Data Transformation and Modeling (22% of the exam) · **Time:** about 40 minutes

**How to run:** attach **Serverless** compute (top right), then run the cells from top to bottom. Each part asks you to predict the result first; the expected results are listed at the end. Everything is created in the `workspace.dea_lab` schema, away from your other work.

From the Learning Hub certification guide: [Data Engineer Associate](https://www.datapreplab.com/hub#certifications/data-engineer-associate) · Independent study resource, not affiliated with or endorsed by Databricks.

## Part 1 · A messy bronze table

In [ ]:
%sql
CREATE SCHEMA IF NOT EXISTS workspace.dea_lab;
USE workspace.dea_lab;
CREATE OR REPLACE TABLE tx_bronze_orders (
  order_id STRING, customer_id STRING, store_id STRING, amount STRING,
  order_ts STRING, channel STRING, status STRING);
INSERT INTO tx_bronze_orders VALUES
  ('1', 'c-10', 's1', '20.50', '2026-11-01 09:15:00', 'WEB',   'OPEN'),
  ('1', 'c-10', 's1', '20.50', '2026-11-01 09:15:00', 'WEB',   'OPEN'),     -- duplicate delivery
  ('2', 'c-11', 's2', '35',    '2026-11-01 10:02:00', 'store', 'SHIPPED'),
  ('3', 'c-12', 's1', '12.00', '2026-11-02 14:40:00', NULL,    'OPEN'),
  (NULL,'c-13', 's2', '99.99', '2026-11-02 15:00:00', 'web',   'OPEN'),     -- no key
  ('4', 'c-10', 's3', '8.25',  '2026-11-03 08:05:00', 'Web',   NULL),
  ('5', NULL,   's2', '15.00', '2026-11-03 11:30:00', 'STORE', 'CANCELLED');
SELECT count(*) AS bronze_rows FROM tx_bronze_orders;

## Part 2 · Clean into silver

**Predict:** how many rows reach silver?

In [ ]:
from pyspark.sql import functions as F

silver = (spark.table("tx_bronze_orders")
    .dropna(subset=["order_id"])                                   # a row without a key is unusable
    .withColumn("order_id", F.col("order_id").cast("int"))
    .withColumn("amount", F.col("amount").cast("decimal(10,2)"))
    .withColumn("order_ts", F.to_timestamp("order_ts"))
    .withColumn("order_date", F.to_date("order_ts"))
    .withColumn("channel", F.lower(F.coalesce("channel", F.lit("unknown"))))
    .dropDuplicates(["order_id"]))
silver.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("tx_silver_orders")
display(spark.table("tx_silver_orders").orderBy("order_id"))

Bronze stays exactly as it arrived, so you can always rebuild silver with better rules.

## Part 3 · Count nulls the right way

**Predict:** the three numbers below.

In [ ]:
%sql
SELECT count(*)                     AS orders,
       count(customer_id)           AS with_customer,
       count_if(customer_id IS NULL) AS missing_customer,
       count(*) - count(status)     AS missing_status
FROM tx_silver_orders;

## Part 4 · Join, union and aggregate

In [ ]:
stores = spark.createDataFrame([("s1", "London"), ("s2", "Dublin"), ("s3", "Paris")], "store_id STRING, city STRING")
enriched = spark.table("tx_silver_orders").join(F.broadcast(stores), "store_id", "left")

# Two monthly extracts with the same columns in a different order.
october = spark.createDataFrame([(100, "London", 10.0)], "order_id INT, city STRING, amount DOUBLE")
november = enriched.select("city", "order_id", F.col("amount").cast("double").alias("amount"))
by_name = october.unionByName(november)
print("unionByName rows:", by_name.count())

display(by_name.groupBy("city").agg(
    F.count("*").alias("orders"),
    F.round(F.sum("amount"), 2).alias("revenue"),
    F.approx_count_distinct("order_id").alias("distinct_orders")).orderBy("city"))

`union` matches columns **by position**: with these two DataFrames it would put city names into `order_id`. `unionByName` matches by name. Neither removes duplicates; add `.distinct()` for SQL `UNION` behaviour.

## Part 5 · Arrays in SQL

In [ ]:
%sql
SELECT tags,
       size(tags)                             AS how_many,
       array_contains(tags, 'gift')           AS is_gift,
       transform(tags, t -> upper(t))         AS upper_tags,
       filter(tags, t -> t <> 'promo')        AS without_promo
FROM VALUES (array('gift', 'promo')), (array('repeat')) AS t(tags);

## Part 6 · MERGE versus INSERT INTO

In [ ]:
%sql
CREATE OR REPLACE TABLE tx_dim_customer (customer_id STRING, tier STRING, updated_at DATE);
INSERT INTO tx_dim_customer VALUES ('c-10', 'basic', DATE'2026-10-01'), ('c-11', 'basic', DATE'2026-10-01');
CREATE OR REPLACE TEMP VIEW tx_customer_feed AS
SELECT * FROM VALUES ('c-10', 'gold', DATE'2026-11-01'), ('c-12', 'basic', DATE'2026-11-01')
AS f(customer_id, tier, updated_at);

**Predict:** run the MERGE twice. How many rows after each run?

In [ ]:
%sql
MERGE INTO tx_dim_customer AS t
USING tx_customer_feed AS s
ON t.customer_id = s.customer_id
WHEN MATCHED AND s.updated_at > t.updated_at THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;
SELECT * FROM tx_dim_customer ORDER BY customer_id;

In [ ]:
%sql
MERGE INTO tx_dim_customer AS t
USING tx_customer_feed AS s
ON t.customer_id = s.customer_id
WHEN MATCHED AND s.updated_at > t.updated_at THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;
SELECT count(*) AS rows_after_second_merge FROM tx_dim_customer;

Now the same feed with INSERT INTO, twice:

In [ ]:
%sql
INSERT INTO tx_dim_customer SELECT * FROM tx_customer_feed;
INSERT INTO tx_dim_customer SELECT * FROM tx_customer_feed;
SELECT customer_id, count(*) AS copies FROM tx_dim_customer GROUP BY customer_id ORDER BY customer_id;

## Part 7 · DELETE and the NULL trap

The task: keep only OPEN orders. **Predict:** which order survives the first DELETE that should not?

In [ ]:
%sql
CREATE OR REPLACE TABLE tx_open_orders AS SELECT * FROM tx_silver_orders;
DELETE FROM tx_open_orders WHERE status <> 'OPEN';
SELECT order_id, status FROM tx_open_orders ORDER BY order_id;

`status <> 'OPEN'` is NULL, not true, when status is NULL, so that row is not deleted. Handle NULL explicitly:

In [ ]:
%sql
DELETE FROM tx_open_orders WHERE status <> 'OPEN' OR status IS NULL;
SELECT order_id, status FROM tx_open_orders ORDER BY order_id;

## Part 8 · Three kinds of gold

In [ ]:
%sql
-- A view: stores only the query, recomputed on every read.
CREATE OR REPLACE VIEW tx_gold_daily_v AS
SELECT order_date, count(*) AS orders, sum(amount) AS revenue FROM tx_silver_orders GROUP BY order_date;

-- A table with a comment (COMMENT comes before AS in CTAS): stored results you refresh yourself.
CREATE OR REPLACE TABLE tx_gold_daily COMMENT 'Daily revenue, rebuilt by the lab' AS
SELECT order_date, count(*) AS orders, sum(amount) AS revenue FROM tx_silver_orders GROUP BY order_date;

SELECT * FROM tx_gold_daily ORDER BY order_date;

In [ ]:
%sql
SHOW TABLES;

A **materialized view** stores results like a table but refreshes itself from its query. Create one in **SQL Editor** with the **Serverless Starter Warehouse** (creating one takes a minute, because Databricks runs a small managed pipeline for it):

```text
CREATE OR REPLACE MATERIALIZED VIEW workspace.dea_lab.tx_gold_daily_mv AS
SELECT order_date, count(*) AS orders, sum(amount) AS revenue
FROM workspace.dea_lab.tx_silver_orders
GROUP BY order_date;

REFRESH MATERIALIZED VIEW workspace.dea_lab.tx_gold_daily_mv;
```

## Part 9 · Optional: the same flow as a declarative pipeline

In **Jobs & Pipelines → Create → ETL pipeline**, choose the `workspace` catalog and `dea_lab` schema, start with an empty SQL file and paste the code below. It reads the files from Lab 2, so run Lab 2 first. Click **Run pipeline** and open the graph.

```text
CREATE OR REFRESH STREAMING TABLE ldp_bronze_orders
AS SELECT * FROM STREAM read_files('/Volumes/workspace/dea_lab/landing/orders/', format => 'json');

CREATE OR REFRESH STREAMING TABLE ldp_silver_orders (
  CONSTRAINT valid_id     EXPECT (order_id IS NOT NULL) ON VIOLATION DROP ROW,
  CONSTRAINT known_amount EXPECT (amount IS NOT NULL)
)
AS SELECT CAST(order_id AS INT) AS order_id, customer, TRY_CAST(amount AS DOUBLE) AS amount, country
FROM STREAM(ldp_bronze_orders);

CREATE OR REFRESH MATERIALIZED VIEW ldp_gold_by_country
AS SELECT country, count(*) AS orders, sum(amount) AS revenue
FROM ldp_silver_orders GROUP BY country;
```

Click **ldp_silver_orders** in the graph and open **Data quality**: `known_amount` warns (the row is kept and counted), `valid_id` would drop rows.

## Expected results

```text
Part 1: 7 bronze rows.
Part 2: 5 silver rows (orders 1-5): the NULL-key row is dropped and the duplicate order 1 removed.
        channel values become web, store or unknown.
Part 3: orders 5, with_customer 4, missing_customer 1, missing_status 1.
Part 4: unionByName returns 6 rows; London has 3 orders, Dublin 2, Paris 1.
Part 5: row 1 -> 2, true, [GIFT, PROMO], [gift]; row 2 -> 1, false, [REPEAT], [repeat].
Part 6: after the first MERGE 3 rows (c-10 becomes gold, c-12 is new). The second MERGE changes
        nothing: still 3 rows. Two INSERT INTO runs add 4 rows: c-10 and c-12 now have 3 copies each.
Part 7: the first DELETE leaves orders 1, 3 and 4 (order 4 has a NULL status). The second leaves 1 and 3.
Part 8: tx_gold_daily has one row per date (2026-11-01, 02 and 03). SHOW TABLES lists the view and tables.
Part 9: ldp_silver_orders keeps 5 rows; the known_amount expectation reports 1 failing record (order 3).
```

## Exam links

- Bronze ingests as-is, silver cleans and deduplicates, gold aggregates for the business.
- INSERT INTO always appends. Use MERGE (or INSERT OVERWRITE) for loads that must be safe to rerun.
- A view does not speed anything up; a materialized view or table stores results.
- Expectations: EXPECT alone warns, ON VIOLATION DROP ROW drops, ON VIOLATION FAIL UPDATE stops the update.